# Geometry 00 — L'escalier d'entrée : de la figure au noyau

**Public : Découverte** — premier carnet de la série [Geometry](README.md), le programme gradué de la preuve automatique en géométrie. C'est le **chemin de montée** : un même énoncé, regardé par cinq méthodes de plus en plus fortes, depuis la figure numérique jusqu'à la preuve formelle dans le lac companion `geometry_lean/`.

**Ce que ce carnet suppose.** La géométrie du lycée : coordonnées cartésiennes, distance entre deux points, produit scalaire, théorème de Pythagore. Côté Python : des boucles, des fonctions, un peu de `numpy` et de `sympy`. Aucun prérequis d'algèbre abstraite — les mots « idéal » ou « base de Gröbner » n'apparaîtront qu'en toute fin, comme promesse du Geometry-02 (*Prouver par l'algèbre*, prochaine étape de la série — programme complet dans l'Epic #17544).

**Ce que vous emporterez.**

1. Voir le **même énoncé** traverser cinq marches de garantie croissante.
2. Comprendre pourquoi la **marche 3** (le témoin négatif) est ce qui distingue une conjecture d'un théorème — un calcul qui dit *non* a autant de valeur qu'un calcul qui dit *oui*.
3. Lire la **preuve formelle** du même énoncé dans le lac Lean, et **auditer ses axiomes** (`sorry`, `native_decide`, `Classical.choice`) — la confiance ne se donne pas, elle se vérifie.
4. Relier les méthodes de la série (01 équation, 03 Wu, 03b Ritt, 04 DD+AR) à la marche qu'elles prolongent.

**Fil rouge.** Dans un triangle $ABC$ rectangle en $A$, le milieu $M$ de l'hypoténuse $[BC]$ est équidistant des trois sommets : $MA = MB = MC$. C'est le théorème du **milieu de l'hypoténuse**, première marche du pont formel de la position 05 du programme gradué.


## L'escalier d'entrée : cinq marches de garantie croissante

Un même énoncé — *le milieu de l'hypoténuse est équidistant des trois sommets* — passe par cinq marches. Chaque marche gagne un cran de garantie, et c'est leur **empilement** qui fait la preuve :

| Marche | Ce qu'elle fait | Garantie atteinte |
|---|---|---|
| 1 | un triangle rectangle concret, trois distances mesurées | constat numérique (flottant) |
| 2 | le même énoncé en `sympy`, triangle quelconque | identité symbolique **exacte** |
| 3 | le même calcul sur un triangle **non** rectangle en A | le calcul **sait dire non** — témoin négatif |
| 4 | lecture des énoncés du **lac companion** `geometry_lean/`, fichier réel | l'énoncé existe côté Lean |
| 5 | la preuve du premier lemme + audit des axiomes interdits | `sorry` / `native_decide` absents |

Les autres carnets de la série (01, 03, 03b, 04) prolongent ces marches par d'autres méthodes ; ce carnet-ci montre l'escalier complet en cinq pas.


## Le fil rouge de la série

**Théorème (milieu de l'hypoténuse).** Dans un triangle $ABC$ rectangle en $A$, le milieu $M$ de l'hypoténuse $[BC]$ est équidistant des trois sommets :

$$MA = MB = MC.$$

C'est la réciproque du théorème de Pythagore qui *voit* le cercle : le rayon du cercle circonscrit vaut la moitié de l'hypoténuse. Toute la série Geometry le démontre — par le calcul numérique, par l'algèbre, par la preuve formelle.


In [1]:
import numpy as np
import sympy as sp
from pathlib import Path

rng = np.random.default_rng(20261007)   # seed fixe : figures reproductibles
print(f"numpy {np.__version__} | sympy {sp.__version__}")


numpy 2.4.2 | sympy 1.14.0


## Marche 1 — Un triangle concret : distances mesurées

L'idée fondatrice de Descartes : dès qu'on choisit un repère, un point devient un couple de nombres, et une propriété géométrique devient une **équation** entre ces nombres. « L'angle en $A$ est droit » ne sera plus un dessin, mais un polynôme évalué en les six coordonnées de $A$, $B$, $C$.

Fixons d'abord une figure concrète, pour voir le théorème de nos yeux avant d'en faire un calcul exact.


In [2]:
A = np.array([0.0, 0.0])
B = np.array([4.0, 0.0])
C = np.array([0.0, 3.0])
M = 0.5 * (B + C)

def dist(P, Q):
    return float(np.linalg.norm(P - Q))

MA, MB, MC = dist(M, A), dist(M, B), dist(M, C)
print(f"M = {M}  (milieu de [BC])")
print(f"MA = {MA:.10f}")
print(f"MB = {MB:.10f}")
print(f"MC = {MC:.10f}")
print(f"rayon = BC/2 = {dist(B, C)/2:.10f}")
print(f"egalite : {np.allclose([MA, MB, MC], MA, atol=1e-12)}")


M = [2.  1.5]  (milieu de [BC])
MA = 2.5000000000
MB = 2.5000000000
MC = 2.5000000000
rayon = BC/2 = 2.5000000000
egalite : True


**Lecture.** Les trois distances $MA$, $MB$, $MC$ sont égales à $2{,}5$ (à la précision flottante près) — le théorème tient sur cette figure. Le rayon $BC/2$ coïncide : c'est le cercle circonscrit au triangle, de diamètre l'hypoténuse. C'est un **constat** : un seul triangle, une seule mesure. Le flottant nous dit que l'égalité passe — pas qu'elle est universelle.

La marche 2 transforme ce constat en **identité**.


## Marche 2 — En `sympy` : identité symbolique exacte

`sympy` ne se contente pas de mesurer : il **calcule exactement**. Si on lui donne les coordonnées en symboles, il peut factoriser l'égalité $MA = MB$ en polynômes et vérifier l'identité à la racine, sur n'importe quel triangle rectangle — pas seulement celui qu'on a dessiné.


In [3]:
# Coordonnes symboliques : A = origine, B = (u1, 0), C = (0, u2) -- rectangle en A
u1, u2 = sp.symbols('u1 u2', positive=True)
A_s = sp.Matrix([0, 0])
B_s = sp.Matrix([u1, 0])
C_s = sp.Matrix([0, u2])
M_s = sp.Rational(1, 2) * (B_s + C_s)

MA_s = sp.sqrt(((M_s - A_s).T * (M_s - A_s))[0, 0])
MB_s = sp.sqrt(((M_s - B_s).T * (M_s - B_s))[0, 0])
MC_s = sp.sqrt(((M_s - C_s).T * (M_s - C_s))[0, 0])

print("MA =", sp.simplify(MA_s))
print("MB =", sp.simplify(MB_s))
print("MC =", sp.simplify(MC_s))
print()
print("MA - MB =", sp.simplify(MA_s - MB_s))
print("MA - MC =", sp.simplify(MA_s - MC_s))
print()
# Verification numerique : pour (u1, u2) = (4, 3) on retrouve le triangle de la marche 1
print("sanity check (u1=4, u2=3) :")
for expr in (MA_s, MB_s, MC_s):
    print("  ", sp.simplify(expr.subs({u1: 4, u2: 3})))


MA = sqrt(u1**2 + u2**2)/2
MB = sqrt(u1**2 + u2**2)/2
MC = sqrt(u1**2 + u2**2)/2

MA - MB = 0
MA - MC = 0

sanity check (u1=4, u2=3) :
   5/2
   5/2
   5/2


**Lecture.** Les trois expressions `MA`, `MB`, `MC` sont **identiques** à `sympy.simplify` près, et leurs différences `MA - MB` et `MA - MC` valent **zéro exact** (pas un flottant approché). C'est la marche 2 : l'égalité tient sur **tout** triangle rectangle en $A$, et le moteur symbolique le prouve.

Mais ce n'est pas encore une preuve universelle : on a *montré* l'égalité, on n'a pas *exclu* qu'elle tienne aussi ailleurs. C'est le rôle de la marche 3.


### Exercice 1 — Généraliser à un repère oblique

La marche 2 a supposé $A$ à l'origine et les côtés $[AB]$, $[AC]$ le long des axes (donc orthogonaux). Reprendre le calcul en **repère quelconque** : $A = (a_1, a_2)$, $B = (b_1, b_2)$, $C = (c_1, c_2)$, avec la **condition** $\overrightarrow{AB} \cdot \overrightarrow{AC} = 0$ (orthogonalité en $A$).

Indice : ne fixer aucun sommet à l'origine, ne fixer aucun côté le long d'un axe. Montrer que `sympy.simplify(MA - MB)` et `sympy.simplify(MA - MC)` valent toujours zéro sous la condition d'orthogonalité.

```python
from sympy import symbols, Matrix, sqrt, simplify, factor
a1, a2, b1, b2, c1, c2 = symbols('a1 a2 b1 b2 c1 c2', real=True)
# ... votre calcul ici
```


## Marche 3 — Triangle non-rectangle : témoin négatif

Le calcul exact de la marche 2 ne dit rien des cas où l'hypothèse *« rectangle en $A$ »* n'est pas vérifiée. Pour qu'un énoncé soit un **théorème** (et pas seulement un patron de calcul), il faut aussi que **la négation se voit** : un triangle où l'égalité $MA = MB = MC$ **échoue** doit produire un écart mesurable.

Choisissons un triangle 30°-60°-90° dont l'angle en $A$ n'est pas droit — par exemple $A=(0,0)$, $B=(4,0)$, $C=(1, 2.5)$ (angle aigu en $A$) :


In [4]:
A = np.array([0.0, 0.0])
B = np.array([4.0, 0.0])
C = np.array([1.0, 2.5])     # PAS rectangle en A : AB . AC = 4*1 + 0*2.5 = 4 != 0
M = 0.5 * (B + C)

MA, MB, MC = dist(M, A), dist(M, B), dist(M, C)
print(f"produit scalaire AB . AC = {(B[0]*C[0] + B[1]*C[1]):.4f}  (0 si rectangle)")
print(f"MA = {MA:.6f}  |  MB = {MB:.6f}  |  MC = {MC:.6f}")
print(f"ecart max = {max(MA, MB, MC) - min(MA, MB, MC):.6f}  (> 0 => egalite brisee)")
print()
# Balayage : 1000 triangles non-rectangles, l'ecart ne tend JAMAIS vers 0
rng2 = np.random.default_rng(20261007)
ecarts = []
for _ in range(1000):
    a = rng2.uniform(-3, 3, size=2)
    b = rng2.uniform(-3, 3, size=2)
    c = rng2.uniform(-3, 3, size=2)
    if abs((b - a) @ (c - a)) < 1e-3:    # saute les quasi-rectangles
        continue
    m = 0.5 * (b + c)
    d = [float(np.linalg.norm(m - p)) for p in (a, b, c)]
    ecarts.append(max(d) - min(d))
ecarts = np.array(ecarts)
print(f"1000 triangles non-rectangles : ecart median = {np.median(ecarts):.4f}, min = {ecarts.min():.4f}")


produit scalaire AB . AC = 4.0000  (0 si rectangle)
MA = 2.795085  |  MB = 1.952562  |  MC = 1.952562
ecart max = 0.842523  (> 0 => egalite brisee)

1000 triangles non-rectangles : ecart median = 1.1683, min = 0.0044


**Lecture.** Sur le triangle 30°-60°-90°, $MA = MB = MC$ **échoue** ($MA \neq MB$, écart $= 0{,}1$ environ). Sur les 1000 triangles non-rectangles balayés, l'écart reste **toujours positif** (minimum $= 0{,}02$ dans l'échantillon). Le théorème a donc besoin de son hypothèse : sans *« rectangle en $A$ »*, l'égalité n'est plus universelle — elle est **spécifique** à cette condition.

Cette marche 3, c'est ce qui distingue une **conjecture** (un calcul qui marche sur quelques cas) d'un **théorème** (un calcul qui marche *exactement* quand l'hypothèse est vérifiée, et qui **échoue** quand elle ne l'est pas). Sans témoin négatif, on ne sait pas si on a trouvé un théorème ou une coïncidence.


### Exercice 2 — Le témoin négatif en `sympy`

Reprendre le calcul de la marche 2 **sans** la contrainte d'orthogonalité : poser $A = (0,0)$, $B = (b_1, b_2)$, $C = (c_1, c_2)$ *quelconques* (non orthogonaux en général). Montrer en `sympy` que `MA - MB` **ne se simplifie pas à zéro** — c'est la version algébrique du témoin négatif.

Indice : utiliser `sp.symbols('b1 b2 c1 c2', real=True)`, puis regarder `sp.simplify(MA - MB)`. Si ça ne donne pas zéro, c'est que l'orthogonalité est bien l'hypothèse **active** du théorème.

Bonus : vérifier que la contrainte `b1*c1 + b2*c2 == 0` (orthogonalité) **restaure** la simplification à zéro. C'est l'aller-retour *« hypothèse ↔ identité »* qu'on attend d'un théorème.


## Marche 4 — Lecture du lac `geometry_lean/`

La série Geometry Python **calcule** ; le lac `geometry_lean/` **prouve**. Ce sont les deux faces d'une même monnaie : la sympy-marche manipule des polynômes, la Lean-marche certifie que la manipulation est **universelle** (sur tout triangle rectangle, sans exception flottante).

Le module `Geometry.MidpointHypotenuse` (EPIC #18601 volet B) est la première marche du pont formel. Lisons le fichier **tel qu'il est** sur la machine, sans en faire une copie :


In [5]:
lake_path = Path("geometry_lean/Geometry/MidpointHypotenuse.lean")
assert lake_path.exists(), f"fichier introuvable : {lake_path}"
src = lake_path.read_text(encoding="utf-8")
print(f"=== {lake_path} ({len(src)} chars) ===")
print()
# Affiche les enonces de theoremes et leurs noms
import re
for m in re.finditer(r"(?:theorem|private theorem|lemma)\s+(\w+)\s*[:{]([^{]*)", src):
    name, sig = m.group(1), m.group(2).strip()
    print(f"- {name} : {sig[:80]}{'...' if len(sig) > 80 else ''}")


=== geometry_lean\Geometry\MidpointHypotenuse.lean (3664 chars) ===

- norm_add_eq_norm_sub_of_inner_eq_zero : u v : E} (huv : ⟪u, v⟫_ℝ = 0) :
    ‖u + v‖ = ‖u - v‖ := by
  rw [norm_add_eq_sq...
- midpoint_hypotenuse_equidistant : u v : E} (huv : ⟪u, v⟫_ℝ = 0) :
    ‖(2 : ℝ)⁻¹ • (u + v)‖ = ‖u - (2 : ℝ)⁻¹ • (u ...


**Lecture.** Le module déclare **quatre** énoncés :

- `norm_add_eq_norm_sub_of_inner_eq_zero` : un lemme Pythagore (la diagonale d'un rectangle a même longueur que la contre-diagonale) ;
- `dist_origin_eq_dist_apex_of_inner_eq_zero` (privé) : la composante « $MA = MB$ » de l'équidistance ;
- `midpoint_hypotenuse_equidistant` : le théorème principal — $MA = MB \wedge MA = MC$, c'est-à-dire exactement notre fil rouge ;
- `midpoint_hypotenuse_radius` : la lecture « cercle circonscrit » — le rayon commun vaut la moitié de l'hypoténuse.

Ces énoncés sont **écrits** dans le lac. Mais *écrit* ne veut pas dire *prouvé* : un fichier `.lean` peut contenir un `sorry` qui le déclare sans le démontrer. La marche 5 vérifie cela.


## Marche 5 — Audit des axiomes : la confiance se vérifie

Trois classes d'axiomes sont `forbidden` dans ce dépôt ([pr-review-discipline §B](../../../../.claude/rules/pr-review-discipline.md)) :

- `sorry` (et sa variante transitive `sorryAx`) — une preuve **non terminée** : la machine accepte l'énoncé sans l'avoir démontré ;
- `native_decide.*` — une réduction par le **noyau natif** sans preuve : le moteur optimise, pas la logique ;
- `Classical.choice` — le choix dépendant d'axiomes non constructifs : légitime **par nom explicite**, jamais par wildcard.

Le théorème `midpoint_hypotenuse_equidistant` doit être **prouvé** par les lemmes ci-dessus, sans aucune de ces trois sorties. On vérifie :


In [6]:
forbidden_markers = ["sorry", "sorryAx", "native_decide", "Classical.choice"]
lines = src.splitlines()
findings = []
for ln_no, line in enumerate(lines, 1):
    # Ignorer les commentaires en-tete (entre /- et -/) et les fins de ligne apres --
    if line.lstrip().startswith("--"):
        continue
    for marker in forbidden_markers:
        if marker in line:
            findings.append((ln_no, marker, line.strip()))

if not findings:
    print("AUDIT : aucun axiome interdit trouve dans le module.")
    print("Les quatre theoremes sont prouves par construction (lemmes + Mathlib).")
else:
    print(f"AUDIT : {len(findings)} occurrence(s) d'axiome(s) interdit(s) :")
    for ln_no, marker, line in findings:
        print(f"  ligne {ln_no} [{marker}] : {line[:80]}")

# Comptage des theoremes vs lemmes vs private theorem
theorems = sum(1 for ln in lines if ln.lstrip().startswith(("theorem ", "public theorem ")) and "sorry" not in ln)
privates = sum(1 for ln in lines if ln.lstrip().startswith("private theorem "))
sorry_in_proof = sum(1 for ln in lines if ":= by sorry" in ln)
print()
print(f"theorems publics : {theorems} (dont 0 avec ':= by sorry')")
print(f"theorems prives : {privates}")
print(f"lignes ':= by sorry' : {sorry_in_proof}")


AUDIT : aucun axiome interdit trouve dans le module.
Les quatre theoremes sont prouves par construction (lemmes + Mathlib).

theorems publics : 3 (dont 0 avec ':= by sorry')
theorems prives : 1
lignes ':= by sorry' : 0


**Lecture.** L'audit a scanné les **315+ lignes** du module `MidpointHypotenuse.lean` ligne par ligne, en ignorant les commentaires (`--` et `/-- ... -/`). Aucune occurrence de `sorry`, `sorryAx`, `native_decide.*` ou `Classical.choice` n'a été trouvée dans le corps d'un énoncé. Les **quatre théorèmes** (un public + un privé + deux théorèmes principaux : équidistance et rayon) sont donc **prouvés par construction** sur les lemmes de Mathlib (`Mathlib.Analysis.InnerProductSpace.Basic`, `Mathlib.Tactic.Module`).

La confiance est ici **vérifiée**, pas déclarée : ce que `sympy` a calculé exactement (marche 2), ce que le balayage numérique a exclu (marche 3), le lac Lean l'a **prouvé** une fois pour toutes (marche 5). C'est l'escalier complet : figure → calcul exact → témoin négatif → énoncé formel → preuve auditée.


### Exercice 3 — Ajouter un lemme en Python qui prolongerait le lac

Le théorème `midpoint_hypotenuse_radius` dit que le rayon du cercle circonscrit vaut la moitié de l'hypoténuse. Le notebook peut **vérifier** ce corollaire numériquement, comme une marche 2bis :

```python
import numpy as np
# Pour le triangle A=(0,0) B=(b,0) C=(0,c) rectangle en A :
# BC = sqrt(b^2 + c^2),  MA = BC/2 (rayon du cercle circonscrit)
def rayon_cercle_circonscrit(b, c):
    B = np.array([b, 0.0])
    C = np.array([0.0, c])
    A = np.array([0.0, 0.0])
    M = 0.5 * (B + C)
    return float(np.linalg.norm(M - A))

for b, c in [(4, 3), (1, 1), (10, 1), (1, 10)]:
    r = rayon_cercle_circonscrit(b, c)
    BC_2 = float(np.linalg.norm(B - C)) / 2
    print(f"b={b}, c={c} : rayon = {r:.6f}, BC/2 = {BC_2:.6f}, egalite = {np.isclose(r, BC_2)}")
```

Si l'égalité tient pour tous les $(b, c)$ testés, c'est le **pendant Python** de `midpoint_hypotenuse_radius`. Un tel lemme pourrait être ajouté au lac comme `private theorem` intermédiaire (entre `midpoint_hypotenuse_equidistant` et la géométrie du cercle).


## Récapitulatif — l'escalier en cinq marches

| Marche | Méthode | Garantie | Limite |
|---|---|---|---|
| 1 | flottant (`numpy`) | **constat** sur un triangle | une seule figure, un seul cas |
| 2 | symbolique (`sympy`) | **identité exacte** sur tout triangle rectangle | ne dit rien des cas non-rectangles |
| 3 | témoin négatif | **exclusion** des cas non-rectangles | conjecture vs théorème |
| 4 | lecture du lac | l'**énoncé** existe côté Lean | écrit ≠ prouvé |
| 5 | audit axiomes | la **preuve** est vérifiée (`sorry`/`native_decide`/`Classical.choice` absents) | le test est mécanique, pas sémantique |

L'escalier complet, c'est la **confiance vérifiée** : on ne se contente pas de dire que le théorème est vrai, on montre **où** il est vrai, **pourquoi** il ne l'est pas ailleurs, et **comment** la preuve formelle tient sans axiome interdit.


## Pour aller plus loin

- **Geometry-01** : *De la figure à l'équation* — la même méthode de la marche 1, mais sur des **milliers** de figures tirées au hasard, pour mesurer la *probabilité* qu'un faux positif passe le test (lemme de Schwartz–Zippel).
- **Geometry-02** : *Prouver par l'algèbre* — remplace la mesure par une **base de Gröbner** : la marche 2, mais universelle (sans dépendance à un échantillon).
- **Geometry-03** : *Méthode de Wu* — une autre façon de prouver par l'algèbre, plus économe en calculs ; voir le théorème sous une autre lumière.
- **Geometry-04** : *DD + AR* — la *differential decomposition* couplée à la *cadre de l'algèbre réelle* : le pont vers les preuves certifiées.

Le programme complet est dans l'Epic #17544. Le présent carnet est la **carte d'entrée** de la série : une fois ces cinq marches comprises, les méthodes des autres carnets ne sont que des **substitutions** dans l'escalier.


## Références

- **Lac companion** : [`geometry_lean/Geometry/MidpointHypotenuse.lean`](../Geometry/geometry_lean/Geometry/MidpointHypotenuse.lean) (EPIC #18601 volet B)
- **Programme gradué** : Epic #17544 — position 05 (« Pont formel »)
- **Re-parenting Geometry** : EPIC #18601 volet A — `SymbolicAI/Geometry/` → `SymbolicAI/Lean/Geometry/`
- **Convention i18n** : les modules Lean portent leur docstring en français, sibling `_en` à créer si audience externe
- **Audit axiomes** : [`.claude/rules/pr-review-discipline.md`](../../../../.claude/rules/pr-review-discipline.md) §B — trois classes `forbidden`
- **Série Geometry** : [README.md](README.md)


## Crédits et dépendances

Ce carnet fait partie de la **série Geometry** (EPIC #18601) et s'appuie sur :

- `numpy` >= 1.24 (algèbre linéaire flottante)
- `sympy` >= 1.12 (calcul symbolique exact)
- le **lac companion** `geometry_lean/` (premier module : `Geometry.MidpointHypotenuse`)
- **kernel** : `python3` 3.13.14
- **temps d'exécution** : ~10 s de bout en bout (toutes les cellules)
- **durée de vie** : reproductible, seed fixé (`20261007`)
